# DiT-XL ImageNet Evaluation on Kaggle
This notebook runs the evaluation pipeline for the Shortcut Models DiT-XL. Kaggle's powerful GPU machines with ample system RAM (up to 30GB) easily sidestep the JAX XLA compiler Out-Of-Memory crashes seen on local WSL 12GB machines.

In [ ]:
# Force upgrade JAX to the latest CUDA 12 version to perfectly align jax, jaxlib, and the jax-cuda12-plugin
# This also natively supports Kaggle's NumPy 2.0 and latest SciPy, wiping out all previous API mismatch errors.
!pip install --upgrade "jax[cuda12]" diffusers einops orbax-checkpoint opencv-python datasets jaxtyping distrax chex ml-collections

## 1. Clone the Repository
Replace the URL below with your GitHub repository. You must `git push` your local code first!

In [ ]:
!git clone https://github.com/ntluongg/shortcut-models.git
%cd shortcut-models

## 2. Mount or Download the Checkpoint
If you uploaded the checkpoint as a Kaggle Dataset, you can link it directly. Otherwise, download it here.

In [ ]:
# Force upgrade JAX to the latest CUDA 12 version to perfectly align jax, jaxlib, and the jax-cuda12-plugin
# This also natively supports Kaggle's NumPy 2.0 and latest SciPy, wiping out all previous API mismatch errors.
!pip install --upgrade "jax[cuda12]" diffusers einops orbax-checkpoint opencv-python datasets jaxtyping distrax chex ml-collections

## 3. Generate ImageNet Samples (NFE = 1)
We use a batch size of 16 to maximize GPU utilization on Kaggle's T4/P100 GPUs.

In [ ]:
!export XLA_PYTHON_CLIENT_ALLOCATOR=platform && \
python train.py \
  --mode inference \
  --model.hidden_size 1152 \
  --model.patch_size 2 \
  --model.depth 28 \
  --model.num_heads 16 \
  --model.mlp_ratio 4 \
  --dataset_name dummy \
  --model.cfg_scale 1.5 \
  --model.class_dropout_prob 0.1 \
  --model.num_classes 1000 \
  --batch_size 16 \
  --inference_generations 50000 \
  --model.train_type shortcut \
  --load_dir "Shortcut Model Checkpoints/imagenet-shortcut2-xl-fulldata-continue200000" \
  --inference_timesteps 1 \
  --samples_dir "eval_samples/nfe_1"

## 4. Compute FID for NFE = 1

In [ ]:
!CUDA_VISIBLE_DEVICES="" python calculate_metrics.py --samples_dir "eval_samples/nfe_1"

## 5. Generate ImageNet Samples (NFE = 2)

In [ ]:
!export XLA_PYTHON_CLIENT_ALLOCATOR=platform && \
python train.py \
  --mode inference \
  --model.hidden_size 1152 \
  --model.patch_size 2 \
  --model.depth 28 \
  --model.num_heads 16 \
  --model.mlp_ratio 4 \
  --dataset_name dummy \
  --model.cfg_scale 1.5 \
  --model.class_dropout_prob 0.1 \
  --model.num_classes 1000 \
  --batch_size 16 \
  --inference_generations 50000 \
  --model.train_type shortcut \
  --load_dir "Shortcut Model Checkpoints/imagenet-shortcut2-xl-fulldata-continue200000" \
  --inference_timesteps 2 \
  --samples_dir "eval_samples/nfe_2"

## 6. Compute FID for NFE = 2

In [ ]:
!CUDA_VISIBLE_DEVICES="" python calculate_metrics.py --samples_dir "eval_samples/nfe_2"